In [1]:
import string
from collections import Counter

def preprocess_text(text, n):
    # 步骤1：文本转小写，清除标点，仅保留字母与空格
    text_lower = text.lower()
    clean_chars = []
    for char in text_lower:
        if char.isalpha() or char == " ":
            clean_chars.append(char)
    clean_text = "".join(clean_chars)

    # 步骤2：按空格分词，过滤空字符串
    word_list = [word for word in clean_text.split() if word.strip()]

    # 步骤3：统计词频，按频率降序构建词汇表，ID从0开始
    word_counter = Counter(word_list)
    # 排序规则：频次高的在前，频次相同按首次出现顺序
    sorted_words = sorted(word_counter.keys(), key=lambda x: (-word_counter[x], word_list.index(x)))
    vocab = {word: idx for idx, word in enumerate(sorted_words)}

    # 步骤4：滑动窗口生成特征与标签，末尾无下一词标签置为None
    features = []
    labels = []
    window_total = len(word_list) - n + 1
    for i in range(window_total):
        window = word_list[i:i + n]
        features.append(window)
        if i + n < len(word_list):
            label = word_list[i + n]
        else:
            label = None
        labels.append(label)

    return vocab, (features, labels)


if __name__ == "__main__":
    # 测试1：题目标准示例
    print("==========测试1：题目示例文本==========")
    text1 = "The time machine"
    n1 = 2
    vocab1, (feat1, label1) = preprocess_text(text1, n1)
    print("词汇表：", vocab1)
    print("特征序列：", feat1)
    print("对应标签：", label1)
    print()

    # 测试2：带标点、重复词汇的额外测试样例
    print("==========测试2：含标点、重复词语文本==========")
    text2 = "Hello! The time, machine. The time flies."
    n2 = 2
    vocab2, (feat2, label2) = preprocess_text(text2, n2)
    print("词汇表：", vocab2)
    print("特征序列：", feat2)
    print("对应标签：", label2)
    print()

    # 测试3：更长文本，n=3窗口测试
    print("==========测试3：长文本，窗口长度n=3==========")
    text3 = "I love deep learning I love python"
    n3 = 3
    vocab3, (feat3, label3) = preprocess_text(text3, n3)
    print("词汇表：", vocab3)
    print("特征序列：", feat3)
    print("对应标签：", label3)

==========测试1：题目示例文本==========
词汇表： {'the': 0, 'time': 1, 'machine': 2}
特征序列： [['the', 'time'], ['time', 'machine']]
对应标签： ['machine', None]

==========测试2：含标点、重复词语文本==========
词汇表： {'the': 0, 'time': 1, 'hello': 2, 'machine': 3, 'flies': 4}
特征序列： [['hello', 'the'], ['the', 'time'], ['time', 'machine'], ['machine', 'the'], ['the', 'time'], ['time', 'flies']]
对应标签： ['time', 'machine', 'the', 'time', 'flies', None]

==========测试3：长文本，窗口长度n=3==========
词汇表： {'i': 0, 'love': 1, 'deep': 2, 'learning': 3, 'python': 4}
特征序列： [['i', 'love', 'deep'], ['love', 'deep', 'learning'], ['deep', 'learning', 'i'], ['learning', 'i', 'love'], ['i', 'love', 'python']]
对应标签： ['learning', 'i', 'love', 'python', None]


In [2]:
import numpy as np

class SimpleRNNCell:
    def __init__(self, input_size, hidden_size):
        self.input_size = input_size
        self.hidden_size = hidden_size
        # 初始化权重（随机正态）
        self.W_hx = np.random.randn(hidden_size, input_size) * 0.01
        self.W_hh = np.random.randn(hidden_size, hidden_size) * 0.01
        self.b_h = np.zeros((hidden_size,))
        
        # 缓存前向传播中间变量，供反向传播使用
        self.cache = None

    def forward(self, x_t, h_prev):
        """
        单步前向传播
        参数：
            x_t: (batch_size, input_size)
            h_prev: (batch_size, hidden_size)
        返回：
            h_t: (batch_size, hidden_size) 当前隐藏状态
        """
        # 计算线性组合 z_t = W_hx @ x_t.T + W_hh @ h_prev.T + b_h
        z_t = np.dot(x_t, self.W_hx.T) + np.dot(h_prev, self.W_hh.T) + self.b_h
        h_t = np.tanh(z_t)
        # 缓存中间变量用于反向传播
        self.cache = (x_t, h_prev, z_t, h_t)
        return h_t

    def backward(self, dh_next):
        """
        单步反向传播，仅计算梯度，不更新权重
        参数：
            dh_next: (batch_size, hidden_size) 损失对h_t的上游梯度
        返回：
            dx_t, dh_prev, dW_hx, dW_hh, db_h
            dx_t: (batch, input_size)
            dh_prev: (batch, hidden_size)
            dW_hx: (hidden, input)
            dW_hh: (hidden, hidden)
            db_h: (hidden,)
        """
        # 取出前向缓存
        x_t, h_prev, z_t, h_t = self.cache
        batch_size = x_t.shape[0]
        
        # tanh导数：dz_t = dh_next * (1 - h_t^2)
        dz_t = dh_next * (1 - np.square(h_t))
        
        # 各参数梯度
        dW_hx = np.dot(dz_t.T, x_t)          # (hidden, input)
        dW_hh = np.dot(dz_t.T, h_prev)       # (hidden, hidden)
        db_h = np.sum(dz_t, axis=0)          # (hidden,) 沿batch求和
        
        # 输入与上一隐藏状态梯度
        dx_t = np.dot(dz_t, self.W_hx)       # (batch, input)
        dh_prev = np.dot(dz_t, self.W_hh)    # (batch, hidden)
        
        return dx_t, dh_prev, dW_hx, dW_hh, db_h


# ---------------------- 测试代码 ----------------------
if __name__ == "__main__":
    # 超参数设置
    batch_size = 4
    input_dim = 3
    hidden_dim = 5

    # 初始化RNN单元
    rnn_cell = SimpleRNNCell(input_size=input_dim, hidden_size=hidden_dim)
    
    # 随机构造输入与上一隐藏状态
    x_t = np.random.randn(batch_size, input_dim)
    h_prev = np.random.randn(batch_size, hidden_dim)
    
    # 1. 前向传播
    h_t = rnn_cell.forward(x_t, h_prev)
    print("===== 前向传播结果 =====")
    print(f"h_t shape: {h_t.shape} 预期({batch_size}, {hidden_dim})")
    print(f"h_t:\n{h_t}\n")

    # 2. 构造上游梯度dh_next（随机模拟损失传回梯度）
    dh_next = np.random.randn(batch_size, hidden_dim)
    dx_t, dh_prev, dW_hx, dW_hh, db_h = rnn_cell.backward(dh_next)

    print("===== 反向传播各梯度维度校验 =====")
    print(f"dx_t shape: {dx_t.shape} 预期({batch_size}, {input_dim})")
    print(f"dh_prev shape: {dh_prev.shape} 预期({batch_size}, {hidden_dim})")
    print(f"dW_hx shape: {dW_hx.shape} 预期({hidden_dim}, {input_dim})")
    print(f"dW_hh shape: {dW_hh.shape} 预期({hidden_dim}, {hidden_dim})")
    print(f"db_h shape: {db_h.shape} 预期({hidden_dim},)\n")

    print("===== 梯度输出示例 =====")
    print("db_h（偏置梯度）：", db_h)

===== 前向传播结果 =====
h_t shape: (4, 5) 预期(4, 5)
h_t:
[[ 0.01442463 -0.01056084 -0.01801784  0.03562492 -0.00164972]
 [ 0.03911151  0.01557327 -0.05166107  0.02111791 -0.01033186]
 [ 0.03286422 -0.04172542  0.0124478  -0.00860741 -0.0004547 ]
 [-0.00499286  0.0169054   0.01784355  0.00259837  0.01333531]]

===== 反向传播各梯度维度校验 =====
dx_t shape: (4, 3) 预期(4, 3)
dh_prev shape: (4, 5) 预期(4, 5)
dW_hx shape: (5, 3) 预期(5, 3)
dW_hh shape: (5, 5) 预期(5, 5)
db_h shape: (5,) 预期(5,)

===== 梯度输出示例 =====
db_h（偏置梯度）： [ 2.51262745 -0.5648324  -0.39205408  0.67877363 -0.95591189]


In [1]:
import torch
import torch.nn as nn

class BiRNNEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        # 单层双向RNN
        self.rnn = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            bidirectional=True,
            batch_first=False  # 匹配输入格式 (seq_len, batch, input_dim)
        )

    def forward(self, X):
        """
        参数：
            X: 输入序列，shape = (seq_len, batch_size, input_dim)
        返回：
            out_concat: 所有时间步拼接隐状态 (seq_len, batch, 2*hidden_dim)
            final_concat: 序列全局表示 (batch, 2*hidden_dim)
        """
        # rnn_output: (seq_len, batch, 2*hidden_dim) 逐时间步前后向已拼接
        # hn: (num_layers*2, batch, hidden_dim)
        rnn_output, hn = self.rnn(X)
        
        # 1. 每个时间步拼接隐状态，直接使用rnn_output即可满足要求
        out_concat = rnn_output
        
        # 2. 拼接最终隐状态：前向最后一步 + 反向第一步
        h_forward = hn[0]   # (batch, hidden_dim) 前向最后时刻
        h_backward = hn[1]  # (batch, hidden_dim) 反向最开始时刻
        final_concat = torch.cat([h_forward, h_backward], dim=-1)  # (batch, 2*hidden_dim)

        return out_concat, final_concat


# -------------------------- 测试代码 --------------------------
if __name__ == "__main__":
    # 超参数
    seq_len = 10    # 序列长度
    batch_size = 4  # 批次大小
    input_dim = 3   # 输入特征维度
    hidden_dim = 5  # 单向隐藏层维度

    # 初始化编码器
    encoder = BiRNNEncoder(input_dim=input_dim, hidden_dim=hidden_dim)
    
    # 构造输入 X: (seq_len, batch, input_dim)
    X = torch.randn(seq_len, batch_size, input_dim)
    
    # 前向传播
    out_all_step, seq_repr = encoder(X)

    print("===== 输出维度校验 =====")
    print(f"输入X shape: {X.shape}  预期({seq_len}, {batch_size}, {input_dim})")
    print(f"各时间步拼接隐状态 out_concat shape: {out_all_step.shape} 预期({seq_len}, {batch_size}, {2*hidden_dim})")
    print(f"序列全局表示 final_concat shape: {seq_repr.shape} 预期({batch_size}, {2*hidden_dim})")
    print("\n===== 输出示例 =====")
    print("序列表示（前向+反向最终拼接）：")
    print(seq_repr)

===== 输出维度校验 =====
输入X shape: torch.Size([10, 4, 3])  预期(10, 4, 3)
各时间步拼接隐状态 out_concat shape: torch.Size([10, 4, 10]) 预期(10, 4, 10)
序列全局表示 final_concat shape: torch.Size([4, 10]) 预期(4, 10)

===== 输出示例 =====
序列表示（前向+反向最终拼接）：
tensor([[-0.3290,  0.0677,  0.2061, -0.5510,  0.2429,  0.5533, -0.7012,  0.3446,
          0.6580,  0.5700],
        [-0.6591, -0.3435, -0.4869,  0.7141, -0.4307,  0.4426,  0.0799,  0.1315,
          0.3321,  0.5254],
        [-0.8728, -0.1447, -0.3590,  0.4474, -0.8330,  0.8164, -0.3193,  0.3734,
          0.8536,  0.5429],
        [-0.2831, -0.7386,  0.8560, -0.3733, -0.7049,  0.7296,  0.1017,  0.3317,
          0.8226,  0.4508]], grad_fn=<CatBackward0>)


In [2]:
import torch
import torch.nn.functional as F

def cbow_forward_loss(batch_contexts, target_words, W, W_out):
    """
    实现CBOW前向传播 + 完整softmax交叉熵损失
    :param batch_contexts: tensor, shape [batch_size, context_size]
                           一批样本的上下文词索引，每个样本context_size个词
    :param target_words: tensor, shape [batch_size]
                           每个样本对应的中心词真实索引
    :param W: 嵌入权重矩阵, shape [V, d]  V词汇总量, d嵌入维度
    :param W_out: 输出层权重, shape [d, V]
    :return: loss: 标量tensor，批次平均交叉熵损失
    """
    batch_size, context_size = batch_contexts.shape
    V, d = W.shape

    # 1. 上下文词嵌入查表 [batch, context_size, d]
    context_embeds = W[batch_contexts]

    # 2. 平均上下文向量得到隐藏层 h [batch, d]
    h = torch.mean(context_embeds, dim=1)

    # 3. 计算输出logits [batch, V]
    logits = torch.matmul(h, W_out)

    # 4. 完整softmax + 交叉熵损失（内部自动softmax，输入原始logits）
    loss = F.cross_entropy(logits, target_words)

    return loss


# ------------------- 测试示例 -------------------
if __name__ == "__main__":
    # 超参数设置
    V = 10    # 词汇表大小
    d = 3     # 嵌入维度
    batch_size = 2
    context_size = 4  # 每个样本4个上下文词

    # 构造输入：2个样本，每个4个上下文词索引
    batch_contexts = torch.tensor([
        [1, 3, 5, 7],
        [2, 4, 6, 8]
    ])
    # 两个样本对应的中心词标签
    target_words = torch.tensor([0, 9])

    # 随机初始化权重矩阵
    W = torch.randn(V, d, requires_grad=True)       # [V, d]
    W_out = torch.randn(d, V, requires_grad=True)   # [d, V]

    # 前向传播计算损失
    loss_val = cbow_forward_loss(batch_contexts, target_words, W, W_out)
    print("批次平均交叉熵损失值：", loss_val.item())

    # 验证梯度可回传（训练可用）
    loss_val.backward()
    print("嵌入矩阵W梯度形状：", W.grad.shape)
    print("输出矩阵W_out梯度形状：", W_out.grad.shape)

批次平均交叉熵损失值： 2.516201972961426
嵌入矩阵W梯度形状： torch.Size([10, 3])
输出矩阵W_out梯度形状： torch.Size([3, 10])


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def multi_head_attention_forward(X):
    """
    多头注意力前向传播，固定 num_heads=2, d_model=4
    :param X: 输入张量 shape [seq_len, batch, d_model] = [S, B, 4]
    :return: output 输出张量 shape [S, B, 4] 和输入完全一致
    """
    num_heads = 2
    d_model = 4
    d_k = d_model // num_heads  # d_k = 2

    # 1. 定义投影层：Q/K/V 投影 + 输出融合线性层
    W_q = nn.Linear(d_model, d_model)
    W_k = nn.Linear(d_model, d_model)
    W_v = nn.Linear(d_model, d_model)
    W_o = nn.Linear(d_model, d_model)

    seq_len, batch, _ = X.shape

    # 2. 线性投影得到 Q, K, V [S, B, 4]
    Q = W_q(X)
    K = W_k(X)
    V = W_v(X)

    # 3. 分头：拆分多头，变换维度 [num_heads, S, B, d_k]
    # 先 reshape [S,B,2,2] 再调换维度
    Q = Q.reshape(seq_len, batch, num_heads, d_k).permute(2, 0, 1, 3)
    K = K.reshape(seq_len, batch, num_heads, d_k).permute(2, 0, 1, 3)
    V = V.reshape(seq_len, batch, num_heads, d_k).permute(2, 0, 1, 3)

    # 4. 缩放点积注意力
    scale = torch.sqrt(torch.tensor(d_k, dtype=torch.float32))
    # Q @ K.transpose(-2,-1)  [heads, S, B, d_k] @ [heads, B, S, d_k].T = [heads, S, B, S]
    attn_score = torch.matmul(Q, K.transpose(-2, -1)) / scale
    attn_weight = F.softmax(attn_score, dim=-1)
    attn_out = torch.matmul(attn_weight, V)  # [heads, S, B, d_k]

    # 5. 拼接多头：还原 [S, B, num_heads*d_k] = [S,B,4]
    attn_out = attn_out.permute(1, 2, 0, 3).reshape(seq_len, batch, d_model)

    # 6. 最终输出线性层
    output = W_o(attn_out)

    return output


# ------------------- 测试示例 -------------------
if __name__ == "__main__":
    # 构造输入：seq_len=3, batch=2, d_model=4
    seq_len = 3
    batch_size = 2
    X = torch.randn(seq_len, batch_size, 4)
    print("输入X shape:", X.shape)

    # 前向传播
    out = multi_head_attention_forward(X)
    print("多头注意力输出 shape:", out.shape)
    print("输出与输入形状是否相同：", out.shape == X.shape)

输入X shape: torch.Size([3, 2, 4])
多头注意力输出 shape: torch.Size([3, 2, 4])
输出与输入形状是否相同： True
